# Policy Intensity, Industry Exposure, and R&D Association

This notebook studies how Sichuan manufacturing policy records can be converted into industry-level policy-exposure measures and linked to observed R&D intensity.

The workflow covers six analytical steps: raw-data audit, alternative policy scoring, exposure-panel construction, cross-sectional association checks, estimator diagnostics, and the final two-way fixed-effects estimate. Real policy and R&D data are used for the empirical analysis; simulations are used only to examine estimator behavior.


In [ ]:
import re
import numpy as np
import pandas as pd
import json
from scipy import stats
import matplotlib
matplotlib.use("Agg")
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

# Shared analysis settings.
RANDOM_STATE = 42
INDUSTRY_CODES = [f"C{code}" for code in range(13, 44)]
YEARS = list(range(2019, 2026))
MONTHS_INDEX = pd.period_range("2019-01", "2025-12", freq="M")

MONTH_MAP = {
    "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "May": 5, "Jun": 6,
    "Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12,
}


def load_real_policy_data(path="00_real_policy_data.csv"):
    """Load policy data and parse issue dates."""
    df = pd.read_csv(path, encoding="utf-8-sig")
    df.columns = [c.strip() for c in df.columns]

    # Standardize raw source columns.
    df = df.rename(columns={
        "文号": "policy_id",
        "层级": "admin_level",
        "支持工具": "instrument",
        "补贴率": "subsidy_rate",
        "贴息率": "interest_subsidy_rate",
        "覆盖行业数": "coverage_count",
        "有效期_月": "duration_months",
        "制造业国标中类代码": "industry_codes_raw",
        "发文年月": "issue_ym_raw",
    })

    def parse_ym(value):
        month_text, year_text = value.split("-")
        return pd.Period(
            year=2000 + int(year_text),
            month=MONTH_MAP[month_text],
            freq="M",
        )

    df["issue_period"] = df["issue_ym_raw"].apply(parse_ym)
    df["issue_year"] = df["issue_period"].dt.year
    df["issue_month"] = df["issue_period"].dt.month
    df["is_broad_call"] = df["industry_codes_raw"].str.strip().eq("CALL")
    df["data_type"] = "REAL"
    return df


def load_real_industry_rd(path="01_real_sichuan_industry_rd_2019_2021_2024.csv"):
    """Load industry-level R&D data."""
    df = pd.read_csv(path, encoding="utf-8-sig")
    df.columns = [c.strip() for c in df.columns]
    return df


def load_real_city_rd(path="02_real_sichuan_city_rd_2021_2024.csv"):
    """Load city-level R&D data."""
    df = pd.read_csv(path, encoding="utf-8-sig")
    df.columns = [c.strip() for c in df.columns]
    return df


def expand_targeted_industries(codes_raw):
    """Map detailed manufacturing codes to two-digit industry groups."""
    if pd.isna(codes_raw) or str(codes_raw).strip() == "CALL":
        return []

    codes = [c.strip() for c in str(codes_raw).split(";") if c.strip()]
    return sorted({
        match.group(1)
        for code in codes
        if (match := re.match(r"(C\d{2})", code))
    })


INSTRUMENT_RULE_SCORE = {
    "补贴": 1.2,
    "贴息": 1.0,
    "奖补": 0.8,
}


def _instrument_rule_score(instrument_str):
    """Return the additive rule score for policy instruments."""
    text = str(instrument_str)
    return sum(
        value
        for key, value in INSTRUMENT_RULE_SCORE.items()
        if key in text
    )


def _minmax01(series):
    series = series.astype(float)
    span = series.max() - series.min()
    if span == 0:
        return pd.Series(0.0, index=series.index)
    return (series - series.min()) / span


def _minmax_to_10(series):
    return _minmax01(pd.Series(series)) * 10.0


def compute_policy_scores(df):
    """Compute rule-based, equal-weight, and PCA policy scores on a 0-10 scale."""
    df = df.copy()

    df["comp_level"] = df["admin_level"].map({2: 1.0, 1: 0.5})
    df["comp_instrument"] = df["instrument"].apply(_instrument_rule_score)
    df["comp_subsidy"] = df["subsidy_rate"].astype(float)
    df["comp_interest"] = df["interest_subsidy_rate"].astype(float)
    df["comp_coverage"] = df["coverage_count"].astype(float)
    df["comp_duration"] = df["duration_months"].astype(float)

    component_cols = [
        "comp_level",
        "comp_instrument",
        "comp_subsidy",
        "comp_interest",
        "comp_coverage",
        "comp_duration",
    ]

    # Primary transparent score.
    rule_raw = (
        df["comp_level"] * 1.0
        + df["comp_instrument"] * 1.0
        + df["comp_subsidy"] * 0.5
        + df["comp_interest"] * 1.0
        + df["comp_coverage"] * 0.15
        + df["comp_duration"] * 0.03
    )
    df["P_rule"] = _minmax_to_10(rule_raw)

    normalized = pd.DataFrame({
        col: _minmax01(df[col])
        for col in component_cols
    })
    df["P_equal"] = normalized.mean(axis=1) * 10

    # PCA-based alternative score.
    X = StandardScaler().fit_transform(df[component_cols].astype(float).values)
    pca = PCA(n_components=1, random_state=RANDOM_STATE)
    pc1 = pca.fit_transform(X).ravel()

    if np.corrcoef(pc1, df["comp_instrument"].astype(float))[0, 1] < 0:
        pc1 = -pc1

    df["P_pca"] = _minmax_to_10(pc1)
    df["pca_explained_variance_ratio"] = pca.explained_variance_ratio_[0]
    return df


def build_industry_month_panel(df_scored):
    """Build the balanced industry-month policy exposure panel."""
    industry_idx = pd.Index(INDUSTRY_CODES, name="industry_code")
    month_idx = pd.PeriodIndex(MONTHS_INDEX, name="year_month")

    panel = pd.MultiIndex.from_product(
        [industry_idx, month_idx],
        names=["industry_code", "year_month"],
    ).to_frame(index=False)

    exposure_cols = [
        "P_rule_it",
        "P_equal_it",
        "P_pca_it",
        "targeted_policy_exposure",
        "broad_policy_exposure",
        "policy_count",
    ]
    for col in exposure_cols:
        panel[col] = 0.0

    panel = panel.set_index(["industry_code", "year_month"])

    # Accumulate exposure over active policy months.
    for _, row in df_scored.iterrows():
        start = row["issue_period"]
        end = start + int(row["duration_months"]) - 1
        active_months = [
            month
            for month in pd.period_range(start, end, freq="M")
            if month in MONTHS_INDEX
        ]
        if not active_months:
            continue

        if row["is_broad_call"]:
            target_industries = INDUSTRY_CODES
            is_targeted = False
        else:
            target_industries = expand_targeted_industries(row["industry_codes_raw"])
            is_targeted = True

        for industry in target_industries:
            if industry not in INDUSTRY_CODES:
                continue

            for month in active_months:
                idx = (industry, month)

                if is_targeted:
                    panel.loc[idx, "P_rule_it"] += row["P_rule"]
                    panel.loc[idx, "P_equal_it"] += row["P_equal"]
                    panel.loc[idx, "P_pca_it"] += row["P_pca"]
                    panel.loc[idx, "targeted_policy_exposure"] += row["P_rule"]
                    panel.loc[idx, "policy_count"] += 1
                else:
                    panel.loc[idx, "broad_policy_exposure"] += row["P_rule"]

    panel = panel.reset_index()
    panel["year"] = panel["year_month"].dt.year
    panel["month"] = panel["year_month"].dt.month
    panel = panel.sort_values(
        ["industry_code", "year_month"]
    ).reset_index(drop=True)

    # Create short exposure lags.
    for col in [
        "P_rule_it",
        "targeted_policy_exposure",
        "broad_policy_exposure",
    ]:
        panel[f"{col}_lag1"] = (
            panel.groupby("industry_code")[col].shift(1).fillna(0.0)
        )
        panel[f"{col}_lag2"] = (
            panel.groupby("industry_code")[col].shift(2).fillna(0.0)
        )

    panel["data_type"] = "REAL_DERIVED"
    return panel


def aggregate_to_industry_year(panel_monthly):
    """Aggregate monthly exposure to industry-year level."""
    panel_yearly = (
        panel_monthly.groupby(["industry_code", "year"])
        .agg(
            policy_exposure_rule=("P_rule_it", "sum"),
            policy_exposure_equal=("P_equal_it", "sum"),
            policy_exposure_pca=("P_pca_it", "sum"),
            policy_exposure_broad=("broad_policy_exposure", "sum"),
            policy_count=("policy_count", "sum"),
        )
        .reset_index()
    )
    panel_yearly["policy_exposure_primary"] = panel_yearly["policy_exposure_rule"]
    panel_yearly["data_type"] = "REAL_DERIVED"
    return panel_yearly


try:
    from linearmodels.panel import PanelOLS
except ImportError as exc:
    raise ImportError(
        "linearmodels>=7.0 is required. Install it with "
        "%pip install -U 'linearmodels>=7.0'"
    ) from exc


def fit_cross_section_ols(y, x, robust="HC3", x_name="policy_exposure"):
    """Fit cross-sectional OLS with optional robust standard errors."""
    data = pd.DataFrame({
        "y": pd.Series(y).astype(float).to_numpy(),
        x_name: pd.Series(x).astype(float).to_numpy(),
    }).dropna()

    X = sm.add_constant(data[[x_name]], has_constant="add")
    model = sm.OLS(data["y"], X)

    if robust is None or str(robust).lower() in {"none", "nonrobust"}:
        result = model.fit()
    else:
        result = model.fit(cov_type=str(robust).upper())

    ci = result.conf_int(alpha=0.05).loc[x_name]
    return {
        "coefficient": float(result.params[x_name]),
        "robust_se": float(result.bse[x_name]),
        "p_value": float(result.pvalues[x_name]),
        "ci_low": float(ci.iloc[0]),
        "ci_high": float(ci.iloc[1]),
        "n": int(result.nobs),
        "r_squared": float(result.rsquared),
        "result": result,
    }


def _panel_frame(y, X, industry_id, year_id, x_names=None):
    """Build the MultiIndex frame required by PanelOLS."""
    X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X.reshape(-1, 1)

    if x_names is None:
        x_names = [f"x{i}" for i in range(X.shape[1])]

    frame = pd.DataFrame(X, columns=x_names)
    frame["y"] = np.asarray(y, dtype=float)
    frame["industry_id"] = np.asarray(industry_id)
    frame["year_id"] = np.asarray(year_id)
    frame = frame.set_index(["industry_id", "year_id"]).sort_index()
    return frame, x_names


def fit_panel_fe(
    y,
    X,
    industry_id,
    year_id,
    *,
    entity_effects=True,
    time_effects=True,
    x_names=None,
):
    """Fit PanelOLS with industry-clustered standard errors."""
    frame, x_names = _panel_frame(
        y,
        X,
        industry_id,
        year_id,
        x_names=x_names,
    )

    model = PanelOLS(
        frame["y"],
        frame[x_names],
        entity_effects=entity_effects,
        time_effects=time_effects,
        drop_absorbed=True,
        check_rank=True,
    )
    return model.fit(
        cov_type="clustered",
        cluster_entity=True,
        debiased=True,
    )


def fit_pooled_clustered(y, X, industry_id, x_names=None):
    """Fit pooled OLS with industry-clustered standard errors."""
    X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X.reshape(-1, 1)

    if x_names is None:
        x_names = [f"x{i}" for i in range(X.shape[1])]

    X_df = pd.DataFrame(X, columns=x_names)
    X_df = sm.add_constant(X_df, has_constant="add")

    model = sm.OLS(np.asarray(y, dtype=float), X_df)
    return model.fit(
        cov_type="cluster",
        cov_kwds={
            "groups": np.asarray(industry_id),
            "use_correction": True,
        },
    )


# Simulation coefficient.
BETA_TRUE = 0.45


def make_base_panel(panel_yearly):
    """Prepare the exposure template used in the controlled simulations."""
    base = panel_yearly[
        ["industry_code", "year", "policy_exposure_rule"]
    ].copy()
    base = base.sort_values(
        ["industry_code", "year"]
    ).reset_index(drop=True)

    sd = base["policy_exposure_rule"].std()
    base["P_it"] = (
        base["policy_exposure_rule"] - base["policy_exposure_rule"].mean()
    ) / (sd + 1e-12)

    base["industry_num"] = (
        base["industry_code"].astype("category").cat.codes
    )
    base["year_num"] = (
        base["year"].astype("category").cat.codes
    )
    return base


def simulate_dgp1(base, seed, noise_std=1.0):
    """Homogeneous panel association with industry and year effects."""
    rng = np.random.default_rng(seed)
    n_industries = base["industry_num"].nunique()
    n_years = base["year_num"].nunique()

    alpha_i = rng.normal(0, 1.0, n_industries)
    lambda_t = rng.normal(0, 0.5, n_years)
    eps = rng.normal(0, noise_std, len(base))

    out = base.copy()
    out["y"] = (
        alpha_i[out["industry_num"].to_numpy()]
        + lambda_t[out["year_num"].to_numpy()]
        + BETA_TRUE * out["P_it"].to_numpy()
        + eps
    )
    return out


def simulate_dgp4(base, seed, noise_std=1.0, targeting_strength=0.6):
    """Dynamic-targeting stress test with lagged simulated performance."""
    rng = np.random.default_rng(seed)
    industries = np.sort(base["industry_num"].unique())
    years = np.sort(base["year_num"].unique())

    n_industries = len(industries)
    alpha_i = rng.normal(0, 1.0, n_industries)
    lambda_t = rng.normal(0, 0.5, len(years))
    y_prev = rng.normal(0, 1.0, n_industries)

    rows = []
    for t_idx, year in enumerate(years):
        P_t = (
            rng.normal(0, 1.0, n_industries)
            - targeting_strength * y_prev
        )
        eps = rng.normal(0, noise_std, n_industries)
        y_t = alpha_i + lambda_t[t_idx] + BETA_TRUE * P_t + eps

        for j, industry in enumerate(industries):
            rows.append({
                "industry_num": industry,
                "year_num": year,
                "P_it": P_t[j],
                "y": y_t[j],
            })

        y_prev = y_t

    return pd.DataFrame(rows)


## A. Policy Data Audit

Before constructing any score, the raw Sichuan policy data are checked for time coverage, missingness, administrative level, targeting type, policy instruments, duration, and industry coverage.

The output is a compact audit table together with descriptive figures. These checks summarize the source data and do not modify the policy records used later.


In [ ]:

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK JP",
    "SimHei",
    "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

df = load_real_policy_data()
missing = df.isna().sum()
n_targeted = (~df["is_broad_call"]).sum()
n_broad = df["is_broad_call"].sum()

# Compact audit summary.
audit_summary = pd.DataFrame({
    "metric": [
        "n_policies",
        "year_min",
        "year_max",
        "n_targeted",
        "n_broad_call",
        "n_admin_level_1",
        "n_admin_level_2",
        "missing_values_total",
    ],
    "value": [
        len(df),
        df["issue_year"].min(),
        df["issue_year"].max(),
        n_targeted,
        n_broad,
        (df["admin_level"] == 1).sum(),
        (df["admin_level"] == 2).sum(),
        int(missing.sum()),
    ],
})
audit_summary["data_type"] = "REAL"
audit_summary.to_csv(
    "module1_data_audit_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

print(audit_summary[["metric", "value"]].to_string(index=False))

# Descriptive policy distributions.
year_counts = df["issue_year"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.bar(
    year_counts.index.astype(str),
    year_counts.values,
    color="#4c72b0",
)
ax.set_xlabel("Year")
ax.set_ylabel("Number of Policies")
ax.set_title("Sichuan Industrial Policies by Year")
for x, y in zip(
    year_counts.index.astype(str),
    year_counts.values,
):
    ax.annotate(
        str(y),
        (x, y),
        textcoords="offset points",
        xytext=(0, 4),
        ha="center",
    )
plt.tight_layout()
plt.savefig("fig1_policies_by_year.png", dpi=150)
plt.close()

instrument_labels = {
    "补贴": "Subsidy",
    "贴息": "Interest subsidy",
    "奖补": "Award/subsidy",
    "补贴+奖补": "Subsidy + award",
    "补贴+贴息": "Subsidy + interest subsidy",
    "补贴+贴息+奖补": "Subsidy + interest subsidy + award",
    "贴息+奖补": "Interest subsidy + award",
}
instrument_display = df["instrument"].replace(instrument_labels)
instrument_year = pd.crosstab(
    df["issue_year"],
    instrument_display,
)

fig, ax = plt.subplots(figsize=(9, 5.5))
instrument_year.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    colormap="tab20",
)
ax.set_xlabel("Year")
ax.set_ylabel("Number of Policies")
ax.set_title("Policy Instrument Composition by Year")
ax.legend(
    title="Instrument",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=8,
)
plt.tight_layout()
plt.savefig("fig2_instruments_by_year.png", dpi=150)
plt.close()

fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(
    df["duration_months"],
    bins=15,
    color="#55a868",
    edgecolor="white",
)
ax.set_xlabel("Duration (months)")
ax.set_ylabel("Number of Policies")
ax.set_title("Policy Duration Distribution")
plt.tight_layout()
plt.savefig("fig3_duration_distribution.png", dpi=150)
plt.close()

# Industry-year policy coverage.
rows = []
for _, row in df.iterrows():
    industries = (
        INDUSTRY_CODES
        if row["is_broad_call"]
        else expand_targeted_industries(row["industry_codes_raw"])
    )
    for industry in industries:
        rows.append({
            "industry_code": industry,
            "year": row["issue_year"],
        })

heat_df = pd.DataFrame(rows)
heat_table = pd.crosstab(
    heat_df["industry_code"],
    heat_df["year"],
)
heat_table = heat_table.reindex(
    INDUSTRY_CODES
).fillna(0)

fig, ax = plt.subplots(figsize=(9, 10))
im = ax.imshow(
    heat_table.values,
    aspect="auto",
    cmap="YlOrRd",
)
ax.set_xticks(range(len(heat_table.columns)))
ax.set_xticklabels(
    heat_table.columns,
    rotation=45,
)
ax.set_yticks(range(len(heat_table.index)))
ax.set_yticklabels(
    heat_table.index,
    fontsize=7,
)
ax.set_xlabel("Year")
ax.set_ylabel("Industry Code")
ax.set_title("Industry-Year Policy Count")
plt.colorbar(
    im,
    ax=ax,
    label="Policy Count",
)
plt.tight_layout()
plt.savefig(
    "fig4_industry_year_heatmap.png",
    dpi=150,
)
plt.close()

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.pie(
    [n_targeted, n_broad],
    labels=["Targeted", "Broad (CALL)"],
    autopct="%1.1f%%",
    colors=["#4c72b0", "#c44e52"],
    startangle=90,
)
ax.set_title("Targeted and Broad Policies")
plt.tight_layout()
plt.savefig("fig5_targeted_vs_broad.png", dpi=150)
plt.close()


              metric  value
          n_policies    100
            year_min   2019
            year_max   2025
          n_targeted     74
        n_broad_call     26
     n_admin_level_1     73
     n_admin_level_2     27
missing_values_total      0


## B. Policy Score Construction and Comparison

Each policy is converted to a 0–10 intensity score using three approaches. `P_rule` is the primary transparent rule-based score, while `P_equal` and `P_pca` provide alternative weighting schemes for robustness.

Agreement across the three definitions is assessed using Pearson and Spearman correlations, ranking differences, and overlap among the highest-ranked policies.


In [ ]:


df = compute_policy_scores(load_real_policy_data())
df.to_csv(
    "module1_policy_scored.csv",
    index=False,
    encoding="utf-8-sig",
)

# Compare score agreement in levels and rankings.
score_cols = ["P_rule", "P_equal", "P_pca"]
pearson_corr = df[score_cols].corr(method="pearson")
spearman_corr = df[score_cols].corr(method="spearman")
ranks = df[score_cols].rank(ascending=False)


def topn_overlap(a, b, n):
    top_a = set(df.nlargest(n, a)["policy_id"])
    top_b = set(df.nlargest(n, b)["policy_id"])
    return len(top_a & top_b) / n


pairs = [
    ("P_rule", "P_equal"),
    ("P_rule", "P_pca"),
    ("P_equal", "P_pca"),
]

comparison_rows = []
for a, b in pairs:
    comparison_rows.append({
        "pair": f"{a} vs {b}",
        "pearson_r": pearson_corr.loc[a, b],
        "spearman_rho": spearman_corr.loc[a, b],
        "mean_abs_rank_diff": (ranks[a] - ranks[b]).abs().mean(),
        "top10_overlap": topn_overlap(a, b, 10),
        "top20_overlap": topn_overlap(a, b, 20),
        "data_type": "REAL_DERIVED",
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(
    "policy_score_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    f"PCA explained variance: "
    f"{df['pca_explained_variance_ratio'].iloc[0]:.4f}"
)
print(
    comparison_df[
        ["pair", "pearson_r", "spearman_rho", "top10_overlap"]
    ].round(4).to_string(index=False)
)

fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(
    pearson_corr.values,
    cmap="RdBu_r",
    vmin=0,
    vmax=1,
)
ax.set_xticks(range(3))
ax.set_xticklabels(pearson_corr.columns)
ax.set_yticks(range(3))
ax.set_yticklabels(pearson_corr.index)

for i in range(3):
    for j in range(3):
        ax.text(
            j,
            i,
            f"{pearson_corr.values[i, j]:.2f}",
            ha="center",
            va="center",
        )

ax.set_title("Policy Score Correlations")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(
    "fig6_score_correlation_heatmap.png",
    dpi=150,
)
plt.close()

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, (a, b) in zip(axes, pairs):
    ax.scatter(
        df[a],
        df[b],
        alpha=0.6,
        color="#4c72b0",
    )
    ax.set_xlabel(a)
    ax.set_ylabel(b)
    ax.set_title(
        f"{a} vs {b}\nPearson r={pearson_corr.loc[a, b]:.3f}"
    )

plt.tight_layout()
plt.savefig("fig7_pairwise_scatter.png", dpi=150)
plt.close()

fig, ax = plt.subplots(figsize=(7.5, 5.5))
order = ranks["P_rule"].argsort()
ax.plot(
    range(len(df)),
    ranks["P_rule"].values[order],
    label="P_rule rank",
    marker=".",
    ms=3,
)
ax.plot(
    range(len(df)),
    ranks["P_equal"].values[order],
    label="P_equal rank",
    marker=".",
    ms=3,
)
ax.plot(
    range(len(df)),
    ranks["P_pca"].values[order],
    label="P_pca rank",
    marker=".",
    ms=3,
)
ax.set_xlabel("Policies sorted by P_rule rank")
ax.set_ylabel("Rank (1 = highest intensity)")
ax.set_title("Policy Score Rank Comparison")
ax.legend()
plt.tight_layout()
plt.savefig("fig8_rank_comparison.png", dpi=150)
plt.close()


PCA explained variance: 0.4263
             pair  pearson_r  spearman_rho  top10_overlap
P_rule vs P_equal     0.8981        0.7610            0.7
  P_rule vs P_pca     0.7998        0.6175            0.8
 P_equal vs P_pca     0.8786        0.8116            0.9


## C. Industry Policy Exposure Panel

Scored policies are expanded over their active months and mapped to two-digit manufacturing industries. Targeted policies contribute to industry-specific exposure, while broad `CALL` policies are stored separately.

The resulting balanced monthly panel is then aggregated to industry-year exposure for the econometric analysis.


In [ ]:


# Build monthly and annual exposure panels.
df = compute_policy_scores(load_real_policy_data())

panel_monthly = build_industry_month_panel(df)
assert panel_monthly.shape[0] == 31 * 84, (
    "Expected 2,604 industry-month observations."
)
panel_monthly.to_csv(
    "industry_month_policy_exposure.csv",
    index=False,
    encoding="utf-8-sig",
)

panel_yearly = aggregate_to_industry_year(panel_monthly)
panel_yearly.to_csv(
    "industry_year_policy_exposure.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    f"Monthly panel: {len(panel_monthly):,} rows | "
    f"Yearly panel: {len(panel_yearly):,} rows"
)


Monthly panel: 2,604 rows | Yearly panel: 217 rows


## D. Cross-Sectional Association Check

To examine whether policy exposure contains an observable signal in later innovation activity, cumulative exposure in 2022–2023 is compared with the change in industry R&D intensity from 2021 to 2024.

HC3-robust OLS and Spearman rank correlation are reported for all three policy-score definitions. This is a preliminary association check rather than the main panel estimate.


In [ ]:


df = compute_policy_scores(load_real_policy_data())
panel_yearly = aggregate_to_industry_year(
    build_industry_month_panel(df)
)

# Match 2022–2023 exposure with the 2021–2024 R&D change.
rd = load_real_industry_rd()
rd_wide = rd.pivot(
    index="industry_code",
    columns="year",
    values="rd_intensity_pct",
)


def exposure_window_sum(panel, years, score_col):
    return (
        panel.loc[panel["year"].isin(years)]
        .groupby("industry_code")[score_col]
        .sum()
    )


def build_design(score_col):
    exposure = exposure_window_sum(
        panel_yearly,
        [2022, 2023],
        score_col,
    )
    delta_rd = rd_wide[2024] - rd_wide[2021]

    return pd.DataFrame({
        "industry_code": rd_wide.index,
        "policy_exposure": (
            exposure.reindex(rd_wide.index)
            .fillna(0.0)
            .to_numpy()
        ),
        "delta_rd_intensity": delta_rd.to_numpy(),
    }).dropna()


score_cols = {
    "P_rule": "policy_exposure_rule",
    "P_equal": "policy_exposure_equal",
    "P_pca": "policy_exposure_pca",
}

rows = []
for score_name, score_col in score_cols.items():
    design = build_design(score_col)
    ols_res = fit_cross_section_ols(
        design["delta_rd_intensity"],
        design["policy_exposure"],
        robust="HC3",
    )
    rho, rho_p = stats.spearmanr(
        design["policy_exposure"],
        design["delta_rd_intensity"],
    )

    rows.append({
        "outcome": "delta_rd_intensity_2021_2024",
        "score_method": score_name,
        "exposure_definition": "targeted_2022_2023",
        "coefficient": ols_res["coefficient"],
        "robust_se": ols_res["robust_se"],
        "p_value": ols_res["p_value"],
        "ci_low": ols_res["ci_low"],
        "ci_high": ols_res["ci_high"],
        "n": ols_res["n"],
        "r_squared": ols_res["r_squared"],
        "spearman_rho": rho,
        "spearman_p": rho_p,
        "data_type": "REAL",
    })

main_results_df = pd.DataFrame(rows)
main_results_df.to_csv(
    "module1_real_results.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    main_results_df[
        [
            "score_method",
            "coefficient",
            "robust_se",
            "p_value",
            "spearman_rho",
        ]
    ].round(4).to_string(index=False)
)


score_method  coefficient  robust_se  p_value  spearman_rho
      P_rule       0.0008     0.0006   0.1744        0.2841
     P_equal       0.0005     0.0004   0.1757        0.2905
       P_pca       0.0005     0.0004   0.1838        0.2265


## E. Estimator Diagnostic

Estimator behavior is evaluated using two controlled data-generating processes with a known coefficient, `beta_true = 0.45`. The homogeneous design provides a benchmark, while the dynamic-targeting design introduces feedback from prior performance into policy exposure.

Pooled OLS, industry fixed effects, and two-way fixed effects are compared using bias, RMSE, and 95% confidence-interval coverage. The simulations are used to assess estimator performance, not to estimate the real policy effect.


In [6]:

N_REPS = 300
Z = 1.959963984540054

df = compute_policy_scores(load_real_policy_data())
panel_yearly = aggregate_to_industry_year(
    build_industry_month_panel(df)
)
base = make_base_panel(panel_yearly)


def collect_fit(res, param_name):
    if hasattr(res, "std_errors"):
        return (
            float(res.params[param_name]),
            float(res.std_errors[param_name]),
        )
    return (
        float(res.params[param_name]),
        float(res.bse[param_name]),
    )


def evaluate(draws, beta_true):
    beta = np.asarray(
        [item[0] for item in draws],
        dtype=float,
    )
    se = np.asarray(
        [item[1] for item in draws],
        dtype=float,
    )

    return {
        "n_success": int(len(beta)),
        "bias": float(beta.mean() - beta_true),
        "rmse": float(
            np.sqrt(
                np.mean(
                    (beta - beta_true) ** 2
                )
            )
        ),
        "empirical_sd": float(beta.std(ddof=1)),
        "mean_reported_se": float(se.mean()),
        "ci_coverage_95": float(
            np.mean(
                (beta - Z * se <= beta_true)
                & (beta_true <= beta + Z * se)
            )
        ),
    }


# Compare estimators across both simulation designs.
rows = []

for dgp_name, simulator in [
    ("DGP1_homogeneous", simulate_dgp1),
    ("DGP4_dynamic_targeting", simulate_dgp4),
]:
    draws = {
        "Pooled_OLS": [],
        "Industry_FE": [],
        "Two_Way_FE": [],
    }
    failures = {
        estimator: 0
        for estimator in draws
    }

    for rep in range(N_REPS):
        seed = (
            10000 + rep
            if dgp_name.startswith("DGP1")
            else 40000 + rep
        )
        simulated = simulator(base, seed=seed)

        y = simulated["y"].to_numpy(float)
        X = simulated[["P_it"]].to_numpy(float)
        industry = simulated["industry_num"].to_numpy()
        year = simulated["year_num"].to_numpy()

        try:
            result = fit_pooled_clustered(
                y,
                X,
                industry,
                x_names=["P_it"],
            )
            draws["Pooled_OLS"].append(
                collect_fit(result, "P_it")
            )
        except Exception:
            failures["Pooled_OLS"] += 1

        try:
            result = fit_panel_fe(
                y,
                X,
                industry,
                year,
                entity_effects=True,
                time_effects=False,
                x_names=["P_it"],
            )
            draws["Industry_FE"].append(
                collect_fit(result, "P_it")
            )
        except Exception:
            failures["Industry_FE"] += 1

        try:
            result = fit_panel_fe(
                y,
                X,
                industry,
                year,
                entity_effects=True,
                time_effects=True,
                x_names=["P_it"],
            )
            draws["Two_Way_FE"].append(
                collect_fit(result, "P_it")
            )
        except Exception:
            failures["Two_Way_FE"] += 1

    for estimator, values in draws.items():
        metrics = evaluate(values, BETA_TRUE)
        rows.append({
            "dgp": dgp_name,
            "estimator": estimator,
            "beta_true": BETA_TRUE,
            "n_requested": N_REPS,
            "n_failures": failures[estimator],
            **metrics,
            "data_type": "SIMULATION",
        })

validation = pd.DataFrame(rows)
validation.to_csv(
    "module1_estimator_validation.csv",
    index=False,
    encoding="utf-8-sig",
)

print(
    validation[
        [
            "dgp",
            "estimator",
            "bias",
            "rmse",
            "ci_coverage_95",
        ]
    ].round(4).to_string(index=False)
)


                   dgp   estimator    bias   rmse  ci_coverage_95
      DGP1_homogeneous  Pooled_OLS  0.0081 0.1716          0.8567
      DGP1_homogeneous Industry_FE  0.0070 0.1922          0.5967
      DGP1_homogeneous  Two_Way_FE -0.0045 0.1277          0.8833
DGP4_dynamic_targeting  Pooled_OLS -0.2194 0.2394          0.2967
DGP4_dynamic_targeting Industry_FE  0.0571 0.0895          0.8267
DGP4_dynamic_targeting  Two_Way_FE  0.0470 0.0740          0.8800


## F. Final Two-Way Fixed-Effects Estimate

The final empirical specification uses the real industry-year panel for the years in which R&D intensity is available:

\[
RD_{it} = \alpha_i + \lambda_t + \beta P_{it} + \varepsilon_{it}.
\]

Here, \(P_{it}\) is the standardized annual rule-based targeted-policy exposure. Industry and year fixed effects absorb time-invariant industry heterogeneity and common year shocks, while standard errors are clustered by industry.

The coefficient is interpreted as an observational association rather than a causal policy effect.


In [7]:



df = compute_policy_scores(load_real_policy_data())
panel_yearly = aggregate_to_industry_year(
    build_industry_month_panel(df)
)
rd = load_real_industry_rd()

PRIMARY_POLICY_SCORE = "P_rule"
PRIMARY_EXPOSURE_DEFINITION = "targeted_only_rule_based_annual"
PRIMARY_PANEL_ESTIMATOR = "Two_Way_FE"

# Restrict the panel to years with observed R&D intensity.
real_panel = (
    panel_yearly.loc[
        panel_yearly["year"].isin([2019, 2021, 2024]),
        [
            "industry_code",
            "year",
            "policy_exposure_rule",
        ],
    ]
    .merge(
        rd[
            [
                "year",
                "industry_code",
                "rd_intensity_pct",
            ]
        ],
        on=["industry_code", "year"],
        how="inner",
    )
    .sort_values(["industry_code", "year"])
    .reset_index(drop=True)
)

# Standardize exposure before fixed-effects estimation.
exposure_mean = real_panel["policy_exposure_rule"].mean()
exposure_std = real_panel["policy_exposure_rule"].std()

real_panel["P_it_std"] = (
    real_panel["policy_exposure_rule"] - exposure_mean
) / exposure_std

real_panel["industry_num"] = (
    real_panel["industry_code"]
    .astype("category")
    .cat.codes
)
real_panel["year_num"] = (
    real_panel["year"]
    .astype("category")
    .cat.codes
)

res = fit_panel_fe(
    real_panel["rd_intensity_pct"].to_numpy(float),
    real_panel[["P_it_std"]].to_numpy(float),
    real_panel["industry_num"].to_numpy(),
    real_panel["year_num"].to_numpy(),
    entity_effects=True,
    time_effects=True,
    x_names=["P_it_std"],
)

beta_hat = float(res.params["P_it_std"])
se_hat = float(res.std_errors["P_it_std"])
ci = res.conf_int(level=0.95).loc["P_it_std"]
ci_low = float(ci.iloc[0])
ci_high = float(ci.iloc[1])
n_obs = int(res.nobs)
n_clusters = int(
    real_panel["industry_num"].nunique()
)

print(
    f"N={n_obs}, industries={n_clusters}, "
    f"beta={beta_hat:.4f}, SE={se_hat:.4f}, "
    f"95% CI=[{ci_low:.4f}, {ci_high:.4f}]"
)

selected_model = {
    "primary_policy_score": PRIMARY_POLICY_SCORE,
    "primary_real_exposure_definition": PRIMARY_EXPOSURE_DEFINITION,
    "primary_panel_estimator": PRIMARY_PANEL_ESTIMATOR,
    "beta_hat": beta_hat,
    "beta_standard_error": se_hat,
    "beta_ci_low": ci_low,
    "beta_ci_high": ci_high,
    "estimation_sample": {
        "n_obs": n_obs,
        "n_industries": n_clusters,
        "years": [2019, 2021, 2024],
        "outcome_variable": "rd_intensity_pct",
        "exposure_variable": (
            "policy_exposure_rule "
            "(standardized, targeted-only)"
        ),
    },
    "exposure_standardization": {
        "mean": float(exposure_mean),
        "std": float(exposure_std),
    },
    "selection_rationale": {
        "policy_score": (
            "P_rule is the primary transparent rule-based score; "
            "P_equal and P_pca are robustness checks."
        ),
        "panel_estimator": (
            "Two-Way FE absorbs time-invariant industry effects "
            "and common year shocks."
        ),
    },
    "software": {
        "python_target": "3.12",
        "cross_section_ols": "statsmodels",
        "panel_fe": "linearmodels.PanelOLS",
    },
    "data_type": "REAL",
    "important_caveat": (
        "Observational association, not a causal policy-effect estimate."
    ),
}

with open(
    "module1_selected_model.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        selected_model,
        file,
        ensure_ascii=False,
        indent=2,
    )


N=93, industries=31, beta=0.2655, SE=0.1585, 95% CI=[-0.0517, 0.5827]


## G. Final Visualizations

Two compact figures are retained for final inspection. The first compares estimator RMSE across the controlled simulation designs, and the second shows annual rule-based policy exposure across manufacturing industries.

These figures summarize the estimator diagnostic and the main exposure pattern without duplicating the full result tables.


In [ ]:


validation = pd.read_csv(
    "module1_estimator_validation.csv"
)
plot_df = validation.copy()
plot_df["label"] = (
    plot_df["dgp"].str.replace("_", " ")
    + "\n"
    + plot_df["estimator"].str.replace("_", " ")
)

# Estimator RMSE comparison.
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(plot_df))
ax.bar(x, plot_df["rmse"])
ax.set_xticks(list(x))
ax.set_xticklabels(
    plot_df["label"],
    rotation=35,
    ha="right",
)
ax.set_ylabel("RMSE of Beta Estimate")
ax.set_title("Estimator Diagnostic")
plt.tight_layout()
plt.savefig(
    "module1_estimator_validation_rmse.png",
    dpi=160,
)
plt.show()


C:\Users\gengh\AppData\Local\Temp\ipykernel_21716\3246530103.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:

# Annual rule-based exposure by industry.
heat_table_score = (
    panel_yearly.pivot(
        index="industry_code",
        columns="year",
        values="policy_exposure_rule",
    )
    .fillna(0)
    .reindex(INDUSTRY_CODES)
    .fillna(0)
)

plt.rcParams.update({
    "font.size": 12,
    "font.family": "sans-serif",
})

fig, ax = plt.subplots(figsize=(10, 12))
sns.heatmap(
    heat_table_score,
    cmap="YlOrRd",
    ax=ax,
    cbar_kws={
        "label": "Policy Exposure Score (Rule-based)"
    },
    linewidths=0,
)

ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Industry Code", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.yticks(fontsize=9)

plt.tight_layout()
plt.savefig(
    "fig14_industry_year_heatmap.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close()
